# Plan d.i -- Data Preparation

Builds the single regression-ready analysis frame (1990-2024) that every downstream analysis step (ii-viii) reads from: independently-recomputed PDI/MDI cross-checked against the pre-computed columns, the SHOCK dummy, and log(FDI) as a documented deviation from the literal thesis methodology.

See `docs/2_plan/analysis/i_data_preparation.md` for the full spec.

**Inputs** (`data/processed/`):
- `economic_indicators_yearly.csv`
- `economic_resilience_index.csv`
- `product_diversification_exports_by_category.csv`
- `market_diversification_exports_by_country.csv`

**Outputs** (`outputs/`):
- `analysis_frame_1990_2024.csv`  (35 rows, baseline)
- `analysis_frame_1990_2023.csv`  (34 rows, robustness subset)
- `reconciliation_report.csv`     (ERI/PDI/MDI recomputed vs precomputed, by year)

**Units note (flagged, not silently resolved):** in `market_diversification_exports_by_country.csv` the 21 country columns are raw USD while `total_exports` is in USD millions -- shares are computed after converting country values to USD millions.

In [1]:
from pathlib import Path

import numpy as np
import pandas as pd

ROOT = Path.cwd().resolve().parents[1]
PROCESSED_DIR = ROOT / "data" / "processed"
OUTPUT_DIR = ROOT / "outputs"

MAIN_IN = PROCESSED_DIR / "economic_indicators_yearly.csv"
ERI_IN = PROCESSED_DIR / "economic_resilience_index.csv"
PDI_IN = PROCESSED_DIR / "product_diversification_exports_by_category.csv"
MDI_IN = PROCESSED_DIR / "market_diversification_exports_by_country.csv"

FRAME_2024_OUT = OUTPUT_DIR / "analysis_frame_1990_2024.csv"
FRAME_2023_OUT = OUTPUT_DIR / "analysis_frame_1990_2023.csv"
RECONCILIATION_OUT = OUTPUT_DIR / "reconciliation_report.csv"

STUDY_START, STUDY_END = 1990, 2024
SHOCK_YEARS = {2008, 2009, 2020, 2021, 2022}

# Country columns are raw USD; total_exports is USD millions (see markdown note above).
MDI_COUNTRY_UNIT_SCALE = 1e6

TOTAL_MERCHANDISE_ROW = "Total Merchandise Exports"

## Step 1 -- Load all four processed CSVs and verify row counts / year coverage

In [2]:
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

main_df = pd.read_csv(MAIN_IN)
eri_df = pd.read_csv(ERI_IN)
pdi_raw = pd.read_csv(PDI_IN)
mdi_df = pd.read_csv(MDI_IN)

assert main_df.shape[0] == 35, f"expected 35 rows in main panel, got {main_df.shape[0]}"
assert eri_df.shape[0] == 35, f"expected 35 rows in ERI detail, got {eri_df.shape[0]}"
assert mdi_df.shape[0] == 35, f"expected 35 rows in MDI export data, got {mdi_df.shape[0]}"
assert set(pdi_raw.columns[1:].astype(int)) >= set(range(1986, 2025)), (
    "PDI export data does not cover the expected 1986-2024 span"
)

print(f"main_df: {main_df.shape}, eri_df: {eri_df.shape}, pdi_raw: {pdi_raw.shape}, mdi_df: {mdi_df.shape}")
main_df.head()

main_df: (35, 7), eri_df: (35, 8), pdi_raw: (18, 40), mdi_df: (35, 23)


,year,economic_resilience_index,product_diversification_index,market_diversification_index,inflation_rate_pct,fdi_net_inflows_usd,exchange_rate
0,1990,0.3837,0.7944,0.8506,21.495250,4.335512e+07,40.06292
1,1991,0.3678,0.7668,0.8120,12.185630,4.834917e+07,41.37150
2,1992,0.3391,0.7238,0.7372,11.383440,1.226258e+08,43.82963
3,1993,0.5452,0.7195,0.7371,11.746740,1.944791e+08,48.32217
4,1994,0.6214,0.7318,0.7385,8.448712,1.664129e+08,49.41514


## Step 2 -- Verify ERI independently

Recompute `eri` as the mean of the three normalized components (`n_gdp`, `n_unemployment_inv`, `n_trade_balance_inv`) and cross-check against both `economic_resilience_index.csv`'s own `eri` column and `economic_indicators_yearly.csv`'s `economic_resilience_index`.

In [3]:
def _hhi(shares: pd.DataFrame) -> pd.Series:
    """DIV = 1 - sum(s_ij^2) per row (year)."""
    return 1 - (shares ** 2).sum(axis=1)


def _verify_eri(eri_df: pd.DataFrame, main_df: pd.DataFrame) -> pd.DataFrame:
    recomputed = eri_df[["n_gdp", "n_unemployment_inv", "n_trade_balance_inv"]].mean(axis=1)
    out = eri_df[["year", "eri"]].rename(columns={"eri": "eri_precomputed_detail"})
    out["eri_recomputed"] = recomputed
    out = out.merge(
        main_df[["year", "economic_resilience_index"]],
        on="year", how="left",
    ).rename(columns={"economic_resilience_index": "eri_precomputed_main"})
    out["diff_recomputed_vs_main"] = out["eri_recomputed"] - out["eri_precomputed_main"]
    return out


eri_check = _verify_eri(eri_df, main_df)
print(f"ERI reconciliation: max |recomputed - main| = {eri_check['diff_recomputed_vs_main'].abs().max():.6f}")
eri_check.head()

ERI reconciliation: max |recomputed - main| = 0.000333


,year,eri_precomputed_detail,eri_recomputed,eri_precomputed_main,diff_recomputed_vs_main
0,1990,0.3837,0.383767,0.3837,6.666667e-05
1,1991,0.3678,0.367767,0.3678,-3.333333e-05
2,1992,0.3391,0.339067,0.3391,-3.333333e-05
3,1993,0.5452,0.545167,0.5452,-3.333333e-05
4,1994,0.6214,0.621400,0.6214,1.110223e-16


## Step 3 -- Recompute PDI via HHI and cross-check

Transpose the category-as-rows export data so year is the row index, restrict to 1990-2024, and compute `DIVP = 1 - sum(s_i^2)` per year using `Total Merchandise Exports` as the denominator.

**Decision flagged:** whether the 17 category values sum to the stated total for every year -- checked below with an explicit tolerance rather than silently assumed.

In [4]:
def _recompute_pdi(pdi_raw: pd.DataFrame) -> pd.DataFrame:
    df = pdi_raw.set_index("product_category")
    total = df.loc[TOTAL_MERCHANDISE_ROW]
    categories = df.drop(index=TOTAL_MERCHANDISE_ROW)

    # Flag (not silently resolve): do the 17 category values sum to the
    # stated total for every year?
    residual = categories.sum(axis=0) - total
    max_abs_residual_pct = (residual.abs() / total * 100).max()
    assert max_abs_residual_pct < 0.05, (
        f"PDI categories vs Total Merchandise Exports residual exceeds tolerance: "
        f"{max_abs_residual_pct:.4f}% max -- categories do not sum to the stated "
        f"total closely enough to ignore; treatment must be revisited."
    )
    # Residual is within floating/rounding tolerance (<0.05%) for every year, so
    # it is ignored rather than added as an implicit "unclassified" share.

    shares = categories.divide(total, axis=1).T
    shares.index = shares.index.astype(int)
    shares = shares.loc[STUDY_START:STUDY_END]
    divp = _hhi(shares)
    divp.name = "divp_recomputed"
    return divp.rename_axis("year").reset_index()


divp = _recompute_pdi(pdi_raw)
divp.head()

,year,divp_recomputed
0,1990,0.794432
1,1991,0.766776
2,1992,0.723811
3,1993,0.719521
4,1994,0.731781


## Step 4 -- Recompute MDI via HHI and cross-check

For each of the 21 country columns, compute its share of `total_exports` (after unit conversion), add an explicit "Others" residual bucket, and compute `DIVM = 1 - sum(s_i^2)`.

**Decision flagged:** in some years the listed countries' converted exports exceed `total_exports`, which would make the "Others" residual negative -- those years are surfaced explicitly and the residual is clipped to 0 in the HHI sum rather than silently allowed to go negative.

In [5]:
def _recompute_mdi(mdi_df: pd.DataFrame) -> pd.DataFrame:
    country_cols = [c for c in mdi_df.columns if c not in ("year", "total_exports")]
    df = mdi_df.set_index("year")
    countries_mn = df[country_cols] / MDI_COUNTRY_UNIT_SCALE
    total = df["total_exports"]

    others = total - countries_mn.sum(axis=1)
    # Flag (not silently resolve): in some years the 21 listed countries'
    # exports, once unit-converted, exceed total_exports -- an "Others"
    # residual cannot be negative, so those years are surfaced here rather
    # than clamped silently.
    negative_others_years = others[others < 0].round(2)

    shares = countries_mn.divide(total, axis=0)
    shares["others_residual"] = (others / total).clip(lower=0)
    divm = _hhi(shares)
    divm.name = "divm_recomputed"
    out = divm.rename_axis("year").reset_index()
    out.attrs["negative_others_years"] = negative_others_years.to_dict()
    return out


divm = _recompute_mdi(mdi_df)
negative_others_years = divm.attrs.get("negative_others_years", {})
if negative_others_years:
    print(
        "MDI 'Others' residual is negative (listed countries exceed total_exports) "
        f"in {len(negative_others_years)} year(s), clipped to 0 in the HHI sum: "
        f"{negative_others_years}"
    )
divm.head()

MDI 'Others' residual is negative (listed countries exceed total_exports) in 4 year(s), clipped to 0 in the HHI sum: {1992: -75.05, 1993: -60.93, 1994: -134.12, 1995: -237.03}


,year,divm_recomputed
0,1990,0.830069
1,1991,0.811653
2,1992,0.737246
3,1993,0.737053
4,1994,0.729237


## Reconciliation report

Compare recomputed ERI/PDI/MDI against the precomputed columns and report the discrepancy explicitly rather than picking a "winner".

In [6]:
def _build_reconciliation_report(
    eri_check: pd.DataFrame, divp: pd.DataFrame, divm: pd.DataFrame, main_df: pd.DataFrame,
) -> pd.DataFrame:
    report = main_df[["year", "product_diversification_index", "market_diversification_index"]].copy()
    report = report.merge(eri_check, on="year", how="left")
    report = report.merge(divp, on="year", how="left")
    report = report.merge(divm, on="year", how="left")
    report["diff_divp"] = report["divp_recomputed"] - report["product_diversification_index"]
    report["diff_divm"] = report["divm_recomputed"] - report["market_diversification_index"]
    return report[[
        "year",
        "eri_recomputed", "eri_precomputed_main", "diff_recomputed_vs_main",
        "divp_recomputed", "product_diversification_index", "diff_divp",
        "divm_recomputed", "market_diversification_index", "diff_divm",
    ]]


reconciliation = _build_reconciliation_report(eri_check, divp, divm, main_df)
reconciliation.to_csv(RECONCILIATION_OUT, index=False)
print(f"PDI reconciliation: mean abs diff = {reconciliation['diff_divp'].abs().mean():.6f}, "
      f"max abs diff = {reconciliation['diff_divp'].abs().max():.6f}")
print(f"MDI reconciliation: mean abs diff = {reconciliation['diff_divm'].abs().mean():.6f}, "
      f"max abs diff = {reconciliation['diff_divm'].abs().max():.6f}")
print(f"Written -> {RECONCILIATION_OUT}")
reconciliation.head()

PDI reconciliation: mean abs diff = 0.000022, max abs diff = 0.000053
MDI reconciliation: mean abs diff = 0.065878, max abs diff = 0.131188
Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/reconciliation_report.csv


,year,eri_recomputed,eri_precomputed_main,diff_recomputed_vs_main,divp_recomputed,product_diversification_index,diff_divp,divm_recomputed,market_diversification_index,diff_divm
0,1990,0.383767,0.3837,6.666667e-05,0.794432,0.7944,0.000032,0.830069,0.8506,-0.020531
1,1991,0.367767,0.3678,-3.333333e-05,0.766776,0.7668,-0.000024,0.811653,0.8120,-0.000347
2,1992,0.339067,0.3391,-3.333333e-05,0.723811,0.7238,0.000011,0.737246,0.7372,0.000046
3,1993,0.545167,0.5452,-3.333333e-05,0.719521,0.7195,0.000021,0.737053,0.7371,-0.000047
4,1994,0.621400,0.6214,1.110223e-16,0.731781,0.7318,-0.000019,0.729237,0.7385,-0.009263


## Steps 5-7 -- Assemble the final analysis frame

Construct the SHOCK dummy, add the flagged `log_fdi` deviation, and assemble one row per year (1990-2024) with both recomputed and precomputed PDI/MDI kept side by side for auditability.

In [7]:
frame = main_df.rename(columns={
    "economic_resilience_index": "eri",
    "product_diversification_index": "divp_source",
    "market_diversification_index": "divm_source",
}).copy()
frame = frame.merge(divp, on="year", how="left")
frame = frame.merge(divm, on="year", how="left")

frame["shock"] = frame["year"].isin(SHOCK_YEARS).astype(int)

# log(FDI) is an explicit, flagged deviation from the literal thesis spec (Ch. 3.5, Eq. 3.1
# uses raw FDI): FDI is log-transformed to reduce scale disparity with the bounded indices;
# the model using raw FDI is reported separately as a robustness check (see step viii).
assert (frame["fdi_net_inflows_usd"] > 0).all(), "log(FDI) undefined for zero/negative FDI values"
frame["log_fdi"] = np.log(frame["fdi_net_inflows_usd"])

frame = frame[[
    "year", "eri", "divp_recomputed", "divm_recomputed", "divp_source", "divm_source",
    "inflation_rate_pct", "exchange_rate", "fdi_net_inflows_usd", "log_fdi", "shock",
]].rename(columns={"divp_recomputed": "divp", "divm_recomputed": "divm"})
frame = frame.sort_values("year").reset_index(drop=True)

assert frame.shape[0] == 35, f"expected 35-row baseline frame, got {frame.shape[0]}"
assert frame["shock"].sum() == len(SHOCK_YEARS), "SHOCK dummy does not match the 5 flagged years"

frame.to_csv(FRAME_2024_OUT, index=False)
print(f"Written -> {FRAME_2024_OUT} ({frame.shape[0]} rows x {frame.shape[1]} cols)")
frame.head()

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/analysis_frame_1990_2024.csv (35 rows x 11 cols)


,year,eri,divp,divm,divp_source,divm_source,inflation_rate_pct,exchange_rate,fdi_net_inflows_usd,log_fdi,shock
0,1990,0.3837,0.794432,0.830069,0.7944,0.8506,21.495250,40.06292,4.335512e+07,17.584935,0
1,1991,0.3678,0.766776,0.811653,0.7668,0.8120,12.185630,41.37150,4.834917e+07,17.693960,0
2,1992,0.3391,0.723811,0.737246,0.7238,0.7372,11.383440,43.82963,1.226258e+08,18.624648,0
3,1993,0.5452,0.719521,0.737053,0.7195,0.7371,11.746740,48.32217,1.944791e+08,19.085835,0
4,1994,0.6214,0.731781,0.729237,0.7318,0.7385,8.448712,49.41514,1.664129e+08,18.929983,0


## Step 8 -- Create the 1990-2023 subset (34 rows)

Held for the robustness check in step viii and the explicit "extends the stated 1990-2023 period by one year" framing required in the write-up.

In [8]:
frame_2023 = frame[frame["year"] <= 2023].reset_index(drop=True)
assert frame_2023.shape[0] == 34, f"expected 34-row robustness subset, got {frame_2023.shape[0]}"
frame_2023.to_csv(FRAME_2023_OUT, index=False)
print(f"Written -> {FRAME_2023_OUT} ({frame_2023.shape[0]} rows x {frame_2023.shape[1]} cols)")
frame_2023.tail()

Written -> /Users/mohamedinas/Desktop/SE_projects/11_fathima_research/1_eda_and_modeling/8_interview_prep/outputs/analysis_frame_1990_2023.csv (34 rows x 11 cols)


,year,eri,divp,divm,divp_source,divm_source,inflation_rate_pct,exchange_rate,fdi_net_inflows_usd,log_fdi,shock
29,2019,0.6242,0.744014,0.809771,0.7440,0.9135,3.528394,178.7449,743466231.5,20.426834,0
30,2020,0.4145,0.763879,0.829151,0.7639,0.9092,6.153945,185.5926,434075668.5,19.888729,1
31,2021,0.6622,0.769416,0.827465,0.7694,0.9110,7.014781,198.7643,592289969.9,20.199507,1
32,2022,0.3567,0.755253,0.821047,0.7552,0.9120,49.721100,322.6327,884150491.4,20.600138,1
33,2023,0.4154,0.792818,0.816997,0.7928,0.9284,16.541170,327.5065,713015364.6,20.385014,0
